# STIR-Net V1 — 14 Trackastra trajectories, Stage-9 style

This notebook reproduces the **same Napari trajectory style used in `notebooks/09_visualization.ipynb`**, but the trajectories come directly from the saved **Trackastra** result used as STIR-Net temporal evidence.

The key layer is created exactly in the Stage-9 form:

```python
viewer.add_tracks(
    tracks_array,
    name="Trackastra Tracks - all",
    scale=SCALE_TZYX,
    tail_length=20,
)
```

So the colored lines are the paths the tracked cells move through over time.

It also adds:

- raw volume;
- preprocessed volume;
- binary mask;
- CC instance labels;
- Trackastra tracked masks;
- Trackastra centroids with IDs;
- all Trackastra trajectories;
- new-track trajectories;
- ended-track trajectories;
- boundary-entry / boundary-exit trajectories;
- GT labels as optional reference only.

This notebook is visualization-only. It does not train or modify STIR-Net.


In [ ]:
from pathlib import Path
from importlib import import_module, reload
import json
import pickle

import numpy as np
import pandas as pd

from learned.stirnet.debugging.acceptance.first_overfit import (
    _repo_root,
    _roi_with_all_cells,
)

REPO_ROOT = _repo_root(Path.cwd())

DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

TRACKASTRA_DIR = DATA_DIR / "trackastra"

TARGET_LOCAL_TIME = 2
BOUNDARY_MARGIN_UM = 4.0
SHOW_BOUNDARY_TRACKS = False

print("Repository :", REPO_ROOT)
print("Data       :", DATA_DIR)
print("Trackastra :", TRACKASTRA_DIR)


## 1. Load the exact five-frame Trackastra input/output

In [ ]:
frame_numbers = np.load(
    DATA_DIR / "frame_numbers.npy"
)

raw_movie = np.load(
    DATA_DIR / "raw_movie.npy",
    mmap_mode="r",
)

processed_movie = np.load(
    DATA_DIR / "processed_movie.npy",
    mmap_mode="r",
)

binary_movie = np.load(
    DATA_DIR / "binary_movie.npy",
    mmap_mode="r",
)

instance_movie = np.load(
    DATA_DIR / "instance_movie.npy",
    mmap_mode="r",
)

gt_movie = np.load(
    DATA_DIR / "gt_movie.npy",
    mmap_mode="r",
)

tracked_masks_path = (
    TRACKASTRA_DIR / "masks_tracked.npy"
)

if not tracked_masks_path.exists():
    raise FileNotFoundError(
        f"Missing Trackastra tracked masks: {tracked_masks_path}"
    )

masks_tracked = np.load(
    tracked_masks_path,
    mmap_mode="r",
)

with (
    TRACKASTRA_DIR / "track_graph.pkl"
).open("rb") as handle:
    track_graph = pickle.load(handle)

with (
    DATA_DIR / "metadata.json"
).open("r", encoding="utf-8") as handle:
    metadata = json.load(handle)

SPACING_ZYX_UM = tuple(
    float(v)
    for v in metadata["spacing_zyx_um"]
)

print("Frames        :", frame_numbers.tolist())
print("Raw           :", raw_movie.shape)
print("Processed     :", processed_movie.shape)
print("Binary        :", binary_movie.shape)
print("CC instances  :", instance_movie.shape)
print("Tracked masks :", masks_tracked.shape)
print("GT            :", gt_movie.shape)
print("Trackastra graph nodes:", track_graph.number_of_nodes())
print("Trackastra graph edges:", track_graph.number_of_edges())
print("Spacing ZYX [um]:", SPACING_ZYX_UM)


## 2. Crop all layers to the same all-cell ROI used by STIR-Net

The Trackastra graph coordinates are shifted by the same ROI offset, so the trajectory lines line up with the images and masks.


In [ ]:
roi, roi_low, roi_high = _roi_with_all_cells(
    instance_movie,
    gt_movie,
    np.asarray(SPACING_ZYX_UM, dtype=np.float32),
)

raw = raw_movie[(slice(None),) + roi]
preprocessed = processed_movie[(slice(None),) + roi]
binary_mask = binary_movie[(slice(None),) + roi]
instance_labels = instance_movie[(slice(None),) + roi]
trackastra_labels = masks_tracked[(slice(None),) + roi]
gt_labels = gt_movie[(slice(None),) + roi]

SCALE_TZYX = (1.0, *SPACING_ZYX_UM)

print("ROI low ZYX :", roi_low.tolist())
print("ROI high ZYX:", roi_high.tolist())
print("ROI movie   :", raw.shape)
print("Scale TZYX  :", SCALE_TZYX)


## 3. Convert the saved **Trackastra graph** to Stage-9-style trajectory rows

This uses Trackastra's own `graph_to_napari_tracks()` conversion.

The resulting rows have the same layout expected by Stage 9:

```text
track_id, frame, z, y, x
```

`cell_id` is sampled from the original CC instance labels only so the centroid labels can display which input component a Trackastra detection sits inside.


In [ ]:
from trackastra.tracking import graph_to_napari_tracks

track_array_full, lineage_graph, _ = (
    graph_to_napari_tracks(track_graph)
)

track_array_full = np.asarray(
    track_array_full,
    dtype=np.float64,
)

if track_array_full.ndim != 2 or track_array_full.shape[1] != 5:
    raise RuntimeError(
        "Trackastra graph_to_napari_tracks did not return "
        "[track_id, time, z, y, x]. "
        f"Got {track_array_full.shape}"
    )

tracks = pd.DataFrame(
    track_array_full,
    columns=[
        "track_id",
        "frame",
        "z",
        "y",
        "x",
    ],
)

tracks["track_id"] = tracks["track_id"].astype(np.int64)
tracks["frame"] = tracks["frame"].astype(np.int64)

# Shift full-volume graph coordinates to the STIR-Net ROI.
tracks[["z", "y", "x"]] = (
    tracks[["z", "y", "x"]].to_numpy(dtype=float)
    - roi_low[None].astype(float)
)

# Keep only coordinates that land inside the cropped scene.
shape_zyx = np.asarray(
    instance_labels.shape[-3:],
    dtype=int,
)

coords_zyx = tracks[
    ["z", "y", "x"]
].to_numpy(dtype=float)

inside = np.all(
    (coords_zyx >= 0)
    & (
        coords_zyx
        < shape_zyx[None]
    ),
    axis=1,
)

tracks = tracks.loc[inside].copy().reset_index(drop=True)

# Attach the current-frame CC ID for centroid text/property display.
cell_ids = []

for row in tracks.itertuples(index=False):
    t = int(row.frame)
    zyx = np.rint(
        [row.z, row.y, row.x]
    ).astype(int)

    if (
        0 <= t < instance_labels.shape[0]
        and np.all(zyx >= 0)
        and np.all(zyx < shape_zyx)
    ):
        cell_ids.append(
            int(
                instance_labels[
                    t,
                    zyx[0],
                    zyx[1],
                    zyx[2],
                ]
            )
        )
    else:
        cell_ids.append(-1)

tracks["cell_id"] = np.asarray(
    cell_ids,
    dtype=np.int64,
)

tracks = tracks.sort_values(
    ["track_id", "frame"]
).reset_index(drop=True)

print("Trackastra trajectory rows:", len(tracks))
print("Unique Trackastra tracks  :", tracks.track_id.nunique())

track_lengths = (
    tracks.groupby("track_id")
    .size()
    .sort_values(ascending=False)
)

print()
print("Track length distribution:")
display(track_lengths.describe())

print()
print("Longest Trackastra tracks:")
display(
    track_lengths.head(20).rename("observations").reset_index()
)

display(tracks.head(30))


## 4. Build exactly the arrays used by the Stage-9 Napari viewer

In [ ]:
tracks_array = tracks[
    ["track_id", "frame", "z", "y", "x"]
].to_numpy(dtype=float)

points_array = tracks[
    ["frame", "z", "y", "x"]
].to_numpy(dtype=float)

track_ids = tracks[
    "track_id"
].to_numpy()

cell_ids = tracks[
    "cell_id"
].to_numpy()

print("tracks_array:", tracks_array.shape)
print("points_array:", points_array.shape)

assert tracks_array.shape[1] == 5
assert points_array.shape[1] == 4


## 5. Reproduce Stage-9 new/ended/boundary trajectory groups

The same endpoint helper used by `notebooks/09_visualization.ipynb` is reused here.

Because the first-overfit scene does not carry the old Stage-4 bounding-box table into this notebook, boundary classification falls back to centroid distance, which the helper already supports.


In [ ]:
endpoint_helpers = reload(
    import_module(
        "src.09_visualization.step02_endpoints"
    )
)

# Minimal cells table. This is enough for the helper;
# without bbox columns it uses centroid boundary distance.
cells_for_endpoints = tracks[
    [
        "frame",
        "cell_id",
        "z",
        "y",
        "x",
    ]
].copy()

endpoint_groups = (
    endpoint_helpers.prepare_endpoint_track_groups(
        tracks,
        cells_for_endpoints,
        instance_labels.shape[-3:],
        voxel_size_zyx=SPACING_ZYX_UM,
        boundary_margin_um=BOUNDARY_MARGIN_UM,
    )
)

print(
    "New failure candidates:",
    endpoint_groups.new_failure_tracks.track_id.nunique(),
)
print(
    "Ended failure candidates:",
    endpoint_groups.ended_failure_tracks.track_id.nunique(),
)
print(
    "Boundary entries:",
    endpoint_groups.boundary_entry_tracks.track_id.nunique(),
)
print(
    "Boundary exits:",
    endpoint_groups.boundary_exit_tracks.track_id.nunique(),
)


# 6. Open Napari — **same trajectory style as Notebook 09**

This is the important cell.

The central line is intentionally the same pattern used by `09_visualization.ipynb`:

```python
viewer.add_tracks(
    tracks_array,
    name="Trackastra Tracks - all",
    scale=SCALE_TZYX,
    tail_length=20,
)
```

The viewer starts at the **last frame**. Since the sequence has only five frames and `tail_length=20`, each multi-frame Trackastra trajectory should appear as its full motion trail.

If you move the time slider backward, the visible tail shortens exactly like the trajectories in Stage 9.


In [ ]:
import napari

napari_layers = reload(
    import_module(
        "src.09_visualization.napari_layers"
    )
)

add_track_group = napari_layers.add_track_group

viewer = napari.Viewer(
    ndisplay=3,
    title="Trackastra trajectories - Stage 9 style",
)

raw_contrast_limits = [
    float(np.percentile(raw, 1)),
    float(np.percentile(raw, 99.8)),
]

viewer.add_image(
    raw,
    name="Raw Volume",
    scale=SCALE_TZYX,
    rendering="mip",
    colormap="gray",
    contrast_limits=raw_contrast_limits,
)

viewer.add_image(
    preprocessed,
    name="Preprocessed Volume",
    scale=SCALE_TZYX,
    rendering="mip",
    colormap="gray",
    contrast_limits=(
        float(np.nanmin(preprocessed)),
        float(np.nanmax(preprocessed)),
    ),
    visible=False,
)

viewer.add_labels(
    binary_mask,
    name="Binary Mask",
    scale=SCALE_TZYX,
    visible=False,
)

viewer.add_labels(
    instance_labels,
    name="Instance Labels",
    scale=SCALE_TZYX,
    visible=False,
)

viewer.add_labels(
    trackastra_labels,
    name="Trackastra Tracked Masks",
    scale=SCALE_TZYX,
    visible=False,
)

viewer.add_labels(
    gt_labels,
    name="GT Labels - reference only",
    scale=SCALE_TZYX,
    visible=False,
)

# ------------------------------------------------------------------
# THIS IS THE SAME TYPE OF TRAJECTORY LAYER AS NOTEBOOK 09.
# ------------------------------------------------------------------
all_tracks_layer = viewer.add_tracks(
    tracks_array,
    name="Trackastra Tracks - all",
    scale=SCALE_TZYX,
    tail_length=20,
)

all_tracks_layer.visible = True

all_centers_layer = viewer.add_points(
    points_array,
    name="Trackastra Centroids - all",
    scale=SCALE_TZYX,
    size=4,
    face_color="red",
    properties={
        "track_id": track_ids,
        "cell_id": cell_ids,
    },
    text={
        "string": "{track_id}",
        "size": 8,
        "color": "white",
        "anchor": "center",
    },
)

all_centers_layer.visible = True

# Same diagnostic track groups used by Stage 9.
add_track_group(
    viewer,
    endpoint_groups.ended_failure_tracks,
    track_name="Trackastra Ended Tracks",
    point_name="Trackastra Ended Centroids",
    color="red",
    scale=SCALE_TZYX,
    visible=False,
)

add_track_group(
    viewer,
    endpoint_groups.new_failure_tracks,
    track_name="Trackastra New Tracks",
    point_name="Trackastra New Centroids",
    color="lime",
    scale=SCALE_TZYX,
    visible=False,
)

add_track_group(
    viewer,
    endpoint_groups.boundary_entry_tracks,
    track_name="Trackastra Boundary Entry Tracks",
    point_name="Trackastra Boundary Entry Centroids",
    color="cyan",
    scale=SCALE_TZYX,
    visible=SHOW_BOUNDARY_TRACKS,
)

add_track_group(
    viewer,
    endpoint_groups.boundary_exit_tracks,
    track_name="Trackastra Boundary Exit Tracks",
    point_name="Trackastra Boundary Exit Centroids",
    color="orange",
    scale=SCALE_TZYX,
    visible=SHOW_BOUNDARY_TRACKS,
)

# Open at the last frame so all five frames can appear in the trajectory tails.
viewer.dims.set_current_step(
    0,
    int(raw.shape[0]) - 1,
)

viewer.layers.selection.active = all_tracks_layer

print()
print("Napari opened at local frame", raw.shape[0] - 1)
print()
print("Primary layers to inspect:")
print("  Trackastra Tracks - all")
print("  Trackastra Centroids - all")
print("  Raw Volume")
print()
print(
    "These are the same Napari trajectory/tail objects used in "
    "notebooks/09_visualization.ipynb, but built from Trackastra's saved graph."
)

napari.run()


## 7. Optional: inspect one suspicious Trackastra trajectory numerically

After closing Napari, set `TRACK_ID_TO_INSPECT` to the ID written beside a trajectory.


In [ ]:
TRACK_ID_TO_INSPECT = None

if TRACK_ID_TO_INSPECT is not None:
    selected = tracks[
        tracks["track_id"]
        == int(TRACK_ID_TO_INSPECT)
    ].copy()

    display(selected)
else:
    print(
        "Set TRACK_ID_TO_INSPECT to a trajectory ID if you want its coordinates."
    )


## How to use this for the Notebook-13 history question

At the final frame, keep **`Trackastra Tracks - all`** visible and rotate the 3-D view.

Then:

1. Find the large merged central component.
2. Follow the colored trajectory lines backward through frames 4 → 0.
3. Count how many distinct Trackastra tracks physically enter or pass through that merged region.
4. Compare that visual count with Notebook 13's result that only two temporal hypotheses were projected onto source 9.

If many trajectories clearly enter source 9 but Notebook 13 associated only two, the likely bug is the **history-to-current-component projection/association**.

If only about two Trackastra trajectories actually reach that region, then the limited historical coverage is genuine and the remaining split hypotheses must come from current-frame spatial evidence.
